# CardioScore Validation 02 — Raw MEA Signal Validation

Accept only long-format raw traces that match the repository's raw-trace schema.

In [ ]:

import sys, subprocess, json, hashlib, zipfile, tarfile
import pandas as pd
from pathlib import Path
PIN = "869150cd5fb5ccf155fb066258404bd4df163ade"
REPO = "Virelion-Biotech/Virelion-CardioScore"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/{REPO}.git@{PIN}"], check=True)
print("Installed pinned CardioScore:", PIN)


In [ ]:

from google.colab import files
up = files.upload()
raw = Path(next(iter(up)))
from virelion_cardioscore.io.raw_trace import load_raw_traces_to_feature_table
features = load_raw_traces_to_feature_table(raw)
features.to_csv("/content/cardioscore_validation/derived/raw_mea_features.csv", index=False)
print("Feature table shape:", features.shape)
print(features.head().to_string(index=False))


In [ ]:

from virelion_cardioscore.analysis.pipeline import CardioScorePipeline
p = CardioScorePipeline.from_defaults()
p.config["concentration_response"]["require_min_concentrations_for_scoring"] = False
result = p.run(features)
print(result.summary_table.to_string(index=False))
assert not result.summary_table.empty
print("QC log:")
for line in result.qc_log: print(" ", line)


In [ ]:

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

manifest = {
    "package_commit": PIN,
    "source_raw": str(raw),
    "source_raw_sha256": file_sha256(raw),
    "canonical_features": "/content/cardioscore_validation/derived/raw_mea_features.csv",
    "canonical_features_sha256": file_sha256("/content/cardioscore_validation/derived/raw_mea_features.csv"),
    "manual_annotation_required_for_accuracy": True
}
Path("/content/cardioscore_validation/results").mkdir(parents=True, exist_ok=True)
Path("/content/cardioscore_validation/results/raw_mea_lineage.json").write_text(json.dumps(manifest, indent=2)+"\n")


### Acceptance rule
The repository software is a secondary comparator. Quantitative accuracy should be anchored to blinded human/reference annotations where available.